***Synthetic Data Generation***

To simulate realistic visitor behavior for the Thanksgiving Point Guest Insights project, synthetic data is generated to represent visitor demographics, visit patterns, preferences, and spending behavior. The dataset is designed to reflect realistic relationships between these variables while avoiding the use of private or personally identifiable information. This allows us to explore data analysis and machine learning techniques in a controlled environment. The synthetic dataset will be used to identify patterns, generate insights, and support data-driven recommendations for improving the guest experience.

In [10]:
import numpy as np
import pandas as pd
import json

In [3]:
np.random.seed(42)
n=1200

##Generating factors that might influence the survey responses
guest_id = np.arange(10001,10001+n)
age = np.random.randint(18,75,n)
family_size = np.random.choice([1,2,3,4,5,6,7,8],n,p=[0.1, 0.05, 0.2, 0.25, 0.2, 0.15,0.04,0.01])
first_visit = np.random.choice(["yes", "no"],n,p=[0.35, 0.65])
visit_frequency = np.random.choice(["First visit", "Rarely", "Quarterly", "Monthly", "Weekly"],n,p=[0.35,0.2,0.3,0.1,0.05])
visit_motivation = np.random.choice(["Education", "Family Time", "Entertainment", "Special Event", "Nature", "Exhibits"],n)
venue = np.random.choice(["Dinosaurs","Garden", "Farm", "Curiosity Museum","Events"],n)
education_program = np.random.choice(["Yes","No"],n,p=[0.4,0.6])
spending = np.maximum(0,np.random.normal(15, 15, n))
participated_booth = np.clip(np.random.poisson(4, n),0,5)
satisfaction = np.clip(np.round(3.5+0.25*(participated_booth)+np.random.normal(0,0.7,n)),1,5)
learning_score = np.clip(np.round(3.2+0.6*(education_program == "Yes")+np.random.normal(0,0.8,n)),1,5)
likelihood_return = np.clip(np.round(satisfaction+0.3*(visit_frequency == "Monthly")+np.random.normal(0,0.5,n)),1,5)
likelihood_recommend = np.clip(np.round(satisfaction+np.random.normal(0,0.5,n)),1,5)


In [4]:
## Put the data into a pandas DataFrame and save it to a CSV file
data = pd.DataFrame({
    "guest_id": guest_id,
    "age": age,
    "family_size": family_size,
    "first_visit": first_visit,
    "visit_frequency": visit_frequency,
    "visit_motivation": visit_motivation,
    "venue": venue,
    "education_program": education_program,
    "spending": spending,
    "satisfaction": satisfaction,
    "learning_score": learning_score,
    "likelihood_return": likelihood_return,
    "likelihood_recommend": likelihood_recommend,
    "participated_booth":participated_booth
})

data.to_csv("../data/guest_survey_data.csv", index=False)
data.head()

,guest_id,age,family_size,first_visit,visit_frequency,visit_motivation,venue,education_program,spending,satisfaction,learning_score,likelihood_return,likelihood_recommend,participated_booth
0,10001,56,3,no,First visit,Entertainment,Garden,No,14.617588,3.0,4.0,3.0,3.0,1
1,10002,69,5,no,Quarterly,Entertainment,Curiosity Museum,Yes,11.742618,5.0,3.0,5.0,5.0,5
2,10003,46,4,yes,Quarterly,Exhibits,Farm,No,15.875554,4.0,3.0,4.0,4.0,2
3,10004,32,5,no,Quarterly,Education,Farm,Yes,5.942116,5.0,4.0,5.0,5.0,4
4,10005,60,5,no,First visit,Family Time,Garden,No,9.851911,5.0,2.0,5.0,5.0,4


***Cleaning the data before analysis***
TO MAINTAIN ORGANIZED RESEARCH FILES AND DATASETS WHILE ENSURING DATA AND CONFIDENTIALITY

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   guest_id              1200 non-null   int64  
 1   age                   1200 non-null   int64  
 2   family_size           1200 non-null   int64  
 3   first_visit           1200 non-null   str    
 4   visit_frequency       1200 non-null   str    
 5   visit_motivation      1200 non-null   str    
 6   venue                 1200 non-null   str    
 7   education_program     1200 non-null   str    
 8   spending              1200 non-null   float64
 9   satisfaction          1200 non-null   float64
 10  learning_score        1200 non-null   float64
 11  likelihood_return     1200 non-null   float64
 12  likelihood_recommend  1200 non-null   float64
 13  participated_booth    1200 non-null   int64  
dtypes: float64(5), int64(4), str(5)
memory usage: 131.4 KB


In [6]:
## Make sure there is no null
data.isnull().sum()

guest_id                0
age                     0
family_size             0
first_visit             0
visit_frequency         0
visit_motivation        0
venue                   0
education_program       0
spending                0
satisfaction            0
learning_score          0
likelihood_return       0
likelihood_recommend    0
participated_booth      0
dtype: int64

In [7]:
## Make sure there is no duplication
data.duplicated().sum()

np.int64(0)

In [8]:
## Make sure it's the right range
print(f"age: {data["age"].min()}-{data["age"].max()}")
print(f"satisfaction: {data["satisfaction"].min()}-{data["satisfaction"].max()}")
print(f"learning_score: {data["learning_score"].min()}-{data["learning_score"].max()}")

age: 18-74
satisfaction: 2.0-5.0
learning_score: 1.0-5.0


In [9]:
comments = [
    "We had a wonderful visit and the kids loved exploring everything.",
    "The gardens were beautiful and very relaxing for our family.",
    "My children especially enjoyed the interactive exhibits.",
    "The staff members were friendly and helpful throughout our visit.",
    "We learned a lot and had a great time together.",
    "The education program was very informative and kept the kids engaged.",
    "There were so many things to see that we could not fit everything into one visit.",
    "The museum was clean and well organized.",
    "The exhibits were fun, but some of them felt a little outdated.",
    "We really enjoyed the hands-on activities.",
    "The kids loved the science demonstrations.",
    "We participated in the activity booth and really enjoyed it.",
    "The booth activity was fun and educational for our children.",
    "We decided not to participate in the booth because our kids wanted to see the exhibits.",
    "The booth was one of the highlights of our visit.",
    "My children enjoyed participating in the booth activity.",
    "We did not participate in the booth, but there were plenty of other activities.",
    "The staff at the booth explained the activity very clearly.",
    "I would definitely recommend this place to other families.",
    "We enjoyed our visit, but the admission price felt a little high.",
    "The food options were convenient but somewhat expensive.",
    "We spent more money than expected because there were so many activities.",
    "The kids really enjoyed participating in the activity booth.",
    "The booth activity was a great way for the kids to learn something new.",
    "We skipped the booth because our children were more interested in the gardens.",
    "The employee at the booth was friendly and made the activity enjoyable.",
    "My child learned something new from the booth activity.",
    "The experience was entertaining for both adults and children.",
    "We appreciated how family-friendly the entire facility was.",
    "The bathrooms were clean and easy to find.",
    "There were plenty of places to sit and take a break.",
    "We participated in the booth and thought the activity was well designed.",
    "Some of the exhibits felt crowded, but the booth was still enjoyable.",
    "We loved the variety of activities available.",
    "There was something interesting for every member of our family.",
    "The interactive exhibits were much better than the traditional displays.",
    "The education program was one of the highlights of our visit.",
    "My children stayed engaged for most of the program.",
    "The instructor did an excellent job explaining the activities.",
    "The program was informative, but it was a little too long for younger children.",
    "We would have liked more opportunities for hands-on learning.",
    "The exhibits helped my children understand the subject better.",
    "I was impressed by how much my children learned today.",
    "We did not participate in the booth, but we still had a great experience.",
    "We are already planning another family visit.",
    "This is a great place to spend a Saturday with children.",
    "We had a great experience from beginning to end.",
    "The booth gave our children something fun and educational to do.",
    "We chose not to participate in the booth because we wanted to explore other areas.",
    "The staff did a good job making the booth activity interesting.",
    "We really liked the interactive technology in the exhibits.",
    "Some of the exhibits were difficult to understand without additional explanation.",
    "The signs and directions were easy to follow.",
    "We enjoyed taking pictures throughout the gardens.",
    "The landscaping was beautiful and well maintained.",
    "The gardens were peaceful and provided a nice break from the busy exhibits.",
    "My kids could have stayed at the activity booth all day.",
    "The hands-on activities were appropriate for different age groups.",
    "The activity booth was a nice addition to the experience.",
    "We did not participate in the booth because our children were tired.",
    "The employees made us feel welcome.",
    "Everyone we talked to was friendly and professional.",
    "One staff member went out of their way to help us with the booth activity.",
    "The booth staff seemed knowledgeable and enthusiastic.",
    "We had a positive experience overall.",
    "The attraction was clean and well maintained.",
    "I liked that there were educational opportunities throughout the visit.",
    "My children were excited to tell their grandparents what they learned.",
    "The experience was more educational than I expected.",
    "We participated in the booth and my children really enjoyed the hands-on experience.",
    "The exhibits were interesting but could use more updates.",
    "There was a good balance between education and entertainment.",
    "We appreciated having activities that the whole family could enjoy.",
    "The kids enjoyed the booth activity more than I expected.",
    "The visit exceeded our expectations.",
    "We thought the experience was good but not exceptional.",
    "The booth was interesting, but we did not spend much time there.",
    "We had trouble finding some of the activities, including the booth.",
    "More information about the booth activities would have been helpful.",
    "The food was good, but the prices were higher than expected.",
    "We bought lunch and several snacks during our visit.",
    "We did not spend much beyond the admission price.",
    "The gift shop had a lot of fun items for children.",
    "My kids wanted to buy everything in the gift shop.",
    "We appreciated having several places to purchase food and drinks.",
    "Our family had a memorable experience today.",
    "This was a great educational experience for my children.",
    "We would definitely participate in the booth again on our next visit.",
    "We enjoyed the visit and would like to come back in the future.",
    "I think our children would enjoy coming back when they are older.",
    "We will probably return because there are still areas we have not explored.",
    "The experience was good enough that we would recommend it to friends.",
    "We enjoyed ourselves, although the cost may prevent us from visiting often.",
    "The kids had fun, especially with the hands-on booth activity.",
    "We liked the educational program more than the general exhibits.",
    "The exhibits were our favorite part of the experience.",
    "The gardens were beautiful, but we wished there were more activities outside.",
    "Overall, this was a positive experience for our family.",
    "We would definitely come back and participate in more activities.",
    "Thanksgiving Point provided a fun and educational experience for our family."
]

In [13]:
guest_comments = [{"id": i + 1,"comment": comment}
                  for i, comment in enumerate(comments)]

with open("../data/guest_comments.json", "w", encoding="utf-8") as f:
    json.dump(guest_comments, f, indent=4, ensure_ascii=False)